# CS 435 — Project 3: Medical X-Ray Anomaly Detection
## Phase 1: Dataset Setup & Preprocessing Pipeline

**Goal of this notebook:**  
Load the NIH Chest X-Ray sample dataset, explore the label distribution to confirm class imbalance, preprocess the images (resize + normalize), and build a clean train/validation/test split that is ready to feed into the CNN (Phase 2) and GAN (Phase 3).

**Dataset:** NIH Chest X-rays (sample — ~5,600 images)  
**Source:** https://www.kaggle.com/datasets/nih-chest-xrays/sample  
**Environment:** Local — Apple Silicon Mac, conda env `dl_cs435`, Python 3.10, tensorflow-macos 2.16.2

---
## Step 1: Configure paths

All paths point to your local project directory.  
Data was already downloaded via the Kaggle CLI — no download cell needed.

In [ ]:
import os

# ── Root project directory (resolved from notebook location) ─────────────────
PROJECT_DIR = os.path.abspath("")

# ── Data locations (from Kaggle sample download) ──────────────────────────────
DATA_DIR   = os.path.join(PROJECT_DIR, "data")
IMAGES_DIR = os.path.join(DATA_DIR, "sample", "images")   # flat folder of PNGs
CSV_PATH   = os.path.join(DATA_DIR, "sample", "sample_labels.csv")

# ── Output directories (created fresh if they don't exist) ───────────────────
SPLITS_DIR  = os.path.join(PROJECT_DIR, "splits")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
for d in [SPLITS_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Verify data is reachable ──────────────────────────────────────────────────
images_on_disk = os.listdir(IMAGES_DIR)
print(f"Images found  : {len(images_on_disk)}")
print(f"CSV exists    : {os.path.exists(CSV_PATH)}")
print(f"Splits dir    : {SPLITS_DIR}")
print(f"Results dir   : {RESULTS_DIR}")

---
## Step 2: Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
import cv2
from pathlib import Path

print(f"TensorFlow version : {tf.__version__}")
print(f"GPU (Metal)        : {tf.config.list_physical_devices('GPU')}")

---
## Step 3: Load the metadata CSV

The sample dataset comes with `sample_labels.csv` which contains one row per image.  
Each row has the image filename and a pipe-separated list of disease labels (e.g. `Pneumonia|Effusion`).  
We parse this into a one-hot encoded DataFrame — one binary column per disease class.

In [ ]:
df = pd.read_csv(CSV_PATH)

# Keep only the columns we need
df = df[['Image Index', 'Finding Labels']]
df.columns = ['filename', 'labels']

print(f"Total images in metadata: {len(df)}")
print("\nSample rows:")
df.head(10)

---
## Step 4: Parse labels and build one-hot encoding

The NIH dataset has 14 disease classes plus a `No Finding` label.  
We focus on a **subset of 5 clinically meaningful classes** to keep training manageable:  
`Pneumonia`, `Effusion`, `Cardiomegaly`, `Atelectasis`, and `No Finding`.  

This subset was chosen because:
- These are among the most common and clinically important findings
- They cover a range of class imbalance severities (Cardiomegaly is rare; Effusion is common)
- A 5-class problem is much faster to train and evaluate than 14 classes

In [ ]:
# Define the disease classes we will focus on
TARGET_CLASSES = ['No Finding', 'Atelectasis', 'Effusion', 'Cardiomegaly', 'Pneumonia']

# Parse the pipe-separated label string into a set
# e.g. 'Pneumonia|Effusion' -> {'Pneumonia', 'Effusion'}
df['label_set'] = df['labels'].apply(lambda x: set(x.split('|')))

# One-hot encode: create one binary column per target class
for cls in TARGET_CLASSES:
    df[cls] = df['label_set'].apply(lambda s: 1 if cls in s else 0)

# Keep only rows where at least one of our target classes is present
mask = df[TARGET_CLASSES].sum(axis=1) > 0
df_filtered = df[mask].reset_index(drop=True)

print(f"Images after filtering to target classes: {len(df_filtered)}")
print("\nLabel distribution (# images per class):")
print(df_filtered[TARGET_CLASSES].sum().sort_values(ascending=False))

---
## Step 5: Visualize the class imbalance

This is the core motivation for Phase 3 (the GAN).  
We expect to see that `No Finding` dominates while `Pneumonia` and `Cardiomegaly` are rare.  
This imbalance causes a naive CNN to bias its predictions toward the majority class.

In [ ]:
class_counts = df_filtered[TARGET_CLASSES].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
bars = plt.bar(class_counts.index, class_counts.values,
               color=['#2196F3' if c != 'No Finding' else '#B0BEC5' for c in class_counts.index],
               edgecolor='white', linewidth=0.8)

for bar, count in zip(bars, class_counts.values):
    plt.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 5,
             f'{count:,}',
             ha='center', va='bottom', fontsize=10)

plt.title('Class distribution — NIH Chest X-Ray Sample (target classes only)', fontsize=13)
plt.ylabel('Number of images')
plt.xlabel('Disease class')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'class_distribution.png'), dpi=150)
plt.show()

majority = class_counts.max()
print("\nImbalance ratios (majority class : minority class):")
for cls, cnt in class_counts.items():
    if cnt < majority:
        print(f"  No Finding vs {cls}: {majority/cnt:.1f}x")

---
## Step 6: Add full file paths and verify images exist

In [ ]:
# All images live flat inside sample/images/
df_filtered['filepath'] = df_filtered['filename'].apply(
    lambda fn: os.path.join(IMAGES_DIR, fn)
)

df_filtered['exists'] = df_filtered['filepath'].apply(os.path.exists)

missing = (~df_filtered['exists']).sum()
print(f"Images found on disk  : {df_filtered['exists'].sum():,}")
print(f"Images missing        : {missing:,}")

df_final = df_filtered[df_filtered['exists']].reset_index(drop=True)
print(f"\nFinal usable dataset  : {len(df_final):,} images")

---
## Step 7: Train / Validation / Test split (70 / 15 / 15)

In [ ]:
train_df, temp_df = train_test_split(
    df_final, test_size=0.30, random_state=42,
    stratify=df_final['No Finding']
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42,
    stratify=temp_df['No Finding']
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train set      : {len(train_df):,} images")
print(f"Validation set : {len(val_df):,} images")
print(f"Test set       : {len(test_df):,} images")
print(f"Total          : {len(train_df)+len(val_df)+len(test_df):,} images")

---
## Step 8: Build the image preprocessing function

Each image goes through: load → decode → resize (224×224) → normalize [0, 1].  
We use `tf.image.decode_image` instead of `decode_png` to safely handle both PNG and JPEG files.

In [ ]:
IMG_SIZE    = 224
CHANNELS    = 1
BATCH_SIZE  = 32
NUM_CLASSES = len(TARGET_CLASSES)

def preprocess_image(filepath, label):
    """
    Load → decode (grayscale) → resize (224×224) → normalize [0, 1].
    decode_image handles both PNG and JPEG transparently.
    set_shape is required after decode_image to give TF a known static shape.
    """
    raw   = tf.io.read_file(filepath)
    image = tf.image.decode_image(raw, channels=CHANNELS, expand_animations=False)
    image.set_shape([None, None, CHANNELS])
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

print("Preprocessing function defined.")
print(f"  Output shape : ({IMG_SIZE}, {IMG_SIZE}, {CHANNELS})")
print(f"  Output range : [0.0, 1.0]")

---
## Step 9: Build tf.data Dataset objects

In [ ]:
def build_dataset(dataframe, shuffle=False):
    filepaths = dataframe['filepath'].values
    labels    = dataframe[TARGET_CLASSES].values.astype('float32')
    dataset   = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(dataframe), seed=42)
    dataset = dataset.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


train_dataset = build_dataset(train_df, shuffle=True)
val_dataset   = build_dataset(val_df,   shuffle=False)
test_dataset  = build_dataset(test_df,  shuffle=False)

print("Datasets built successfully.")
print(f"  Train batches      : {len(train_dataset)}")
print(f"  Validation batches : {len(val_dataset)}")
print(f"  Test batches       : {len(test_dataset)}")

---
## Step 10: Verify the pipeline — inspect one batch

In [ ]:
sample_images, sample_labels = next(iter(train_dataset))

print("=== Batch sanity check ===")
print(f"Image batch shape  : {sample_images.shape}")   # expect (32, 224, 224, 1)
print(f"Label batch shape  : {sample_labels.shape}")   # expect (32, 5)
print(f"Image dtype        : {sample_images.dtype}")   # expect float32
print(f"Pixel value range  : [{sample_images.numpy().min():.4f}, {sample_images.numpy().max():.4f}]")
print(f"\nSample labels (first 5 images):")
label_df = pd.DataFrame(sample_labels[:5].numpy(), columns=TARGET_CLASSES)
print(label_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

for i in range(8):
    img     = sample_images[i].numpy().squeeze()
    label   = sample_labels[i].numpy()
    present = [TARGET_CLASSES[j] for j, v in enumerate(label) if v == 1]
    title   = ', '.join(present) if present else 'No Finding'
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(title, fontsize=9)
    axes[i].axis('off')

plt.suptitle('Sample chest X-rays from training set (after preprocessing)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'sample_xrays.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Step 11: Save the DataFrames for Phase 2 and Phase 3

In [ ]:
train_df.to_csv(os.path.join(SPLITS_DIR, 'train.csv'), index=False)
val_df.to_csv(  os.path.join(SPLITS_DIR, 'val.csv'),   index=False)
test_df.to_csv( os.path.join(SPLITS_DIR, 'test.csv'),  index=False)

print("Splits saved:")
print(f"  {SPLITS_DIR}/train.csv  ({len(train_df):,} rows)")
print(f"  {SPLITS_DIR}/val.csv    ({len(val_df):,} rows)")
print(f"  {SPLITS_DIR}/test.csv   ({len(test_df):,} rows)")
print("\nPhase 1 complete. Ready for Phase 2 (CNN baseline).")

---
## Phase 1 Summary

| Step | What we did | Key output |
|------|-------------|------------|
| 1    | Configured local paths | All dirs verified |
| 2    | Imported libraries, confirmed Metal GPU | TF + GPU ready |
| 3    | Loaded `sample_labels.csv` | Raw DataFrame |
| 4    | Parsed multi-hot labels for 5 target classes | `df_filtered` |
| 5    | Visualized class imbalance | `class_distribution.png` |
| 6    | Verified all image files exist on disk | Clean `df_final` |
| 7    | 70/15/15 train/val/test split | `train_df`, `val_df`, `test_df` |
| 8–9  | Built `tf.data` preprocessing pipeline | `train/val/test_dataset` |
| 10   | Sanity-checked shapes, pixel values, visuals | Pipeline confirmed |
| 11   | Saved splits as CSV files | `splits/train.csv`, `val.csv`, `test.csv` |

**Next step → Phase 2:** Build and train the CNN baseline using only the real (unaugmented) data.